# Fashion-MNIST 10 Binary Specialists & Multi-Task Generalist with Pareto Optimization (KD)

Notebook ini berisi alur *workflow* lengkap yang 100% identik dengan eksperimen Multi-Fashion-MNIST Generalist Pareto, dengan modifikasi pada model **Specialist** yang diubah menjadi **10 Binary Specialists (One-vs-All)**.

## 1. Imports, Setup Device & Configuration

In [ ]:
import numpy as np, torch, torch.nn as nn, torch.nn.functional as F
import torchvision, matplotlib.pyplot as plt, random, time, copy
from torchvision import datasets, models
import pandas as pd

dev = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', dev)

CFG = dict(
    N_TRAIN_MULTI = 20000,
    N_VAL_MULTI   = 2000,
    N_TEST_MULTI  = 5000,
    N_SPEC_HARD   = 20000,
    LABELED_FRAC  = 0.05,
    BS            = 128,
    LR            = 1e-3,
    SPEC_EPOCHS   = 3,
    GEN_EPOCHS    = 4,
    T_KD          = 2.0,
    LAM           = 0.7,
    METHODS       = ['ls', 'pcgrad', 'mgda', 'famo'],
    N_PREF_EXTRA  = 0,
    EXPAND_ROUNDS = 1,
    SEED          = 0,
)

def set_seed(s):
    random.seed(s)
    np.random.seed(s)
    torch.manual_seed(s)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(s)

set_seed(CFG['SEED'])

## 2. Data Preparation & Visualization

In [ ]:
CANVAS, DS = 36, 28

# Anda bisa menentukan jumlah tumpukan maksimum atau kisaran jumlah pakaian (K_min sampai K_max)
K_MAX = 4  # Contoh: maksimum 4 pakaian bertumpuk per gambar

# SLOT_POS = ((0, 0), (8, 8))
CLASS_NAMES = (
    'T-shirt/top', 'Trouser', 'Pullover', 'Dress', 'Coat',
    'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot'
)
N_CLASSES = len(CLASS_NAMES)

fashion_tr = datasets.FashionMNIST('./data', train=True, download=True)
fashion_te = datasets.FashionMNIST('./data', train=False, download=True)
all_tr_x, all_tr_y = fashion_tr.data.numpy(), fashion_tr.targets.numpy()
fa_te_x, fa_te_y = fashion_te.data.numpy(), fashion_te.targets.numpy()

split_rng = np.random.RandomState(CFG['SEED'])
perm = split_rng.permutation(len(all_tr_x))
train_ids, val_ids = perm[:54000], perm[54000:]
fa_tr_x, fa_tr_y = all_tr_x[train_ids], all_tr_y[train_ids]
fa_val_x, fa_val_y = all_tr_x[val_ids], all_tr_y[val_ids]

# def make_multi(imgs, labels, n, seed):
#     rng = np.random.RandomState(seed)
#     X = np.zeros((n, CANVAS, CANVAS), dtype=np.uint8)
#     Y = np.zeros((n, K), dtype=np.int64)
#     P = np.broadcast_to(np.asarray(SLOT_POS, np.int64), (n, K, 2)).copy()
#     for i in range(n):
#         idx = rng.randint(0, len(imgs), size=K)
#         for k, j in enumerate(idx):
#             y, x = SLOT_POS[k]
#             patch = X[i, y:y+DS, x:x+DS].astype(np.uint16) + imgs[j].astype(np.uint16)
#             X[i, y:y+DS, x:x+DS] = np.minimum(patch, 255).astype(np.uint8)
#             Y[i, k] = labels[j]
#     return X, Y, P

def make_multi_flexible(imgs, labels, n, k_min=1, k_max=K_MAX, seed=1):
    """
    Membuat dataset komposit bertumpuk dengan:
    - Jumlah pakaian bertumpuk (K) bervariasi secara acak per sampel (antara k_min hingga k_max)
    - Posisi offset (y, x) acak di dalam area kanvas 36x36
    """
    rng = np.random.RandomState(seed)
    
    X = np.zeros((n, CANVAS, CANVAS), dtype=np.uint8)
    
    # Target label: jika jumlah pakaian bervariasi, kita pad dengan -1 untuk slot kosong
    Y = np.full((n, k_max), -1, dtype=np.int64)
    
    # Koordinat Posisi P (n, k_max, 2): simpan (y, x) acak
    P = np.full((n, k_max, 2), -1, dtype=np.int64)
    
    # Rentang koordinat maksimum agar item 28x28 tetap masuk di dalam kanvas 36x36
    max_offset = CANVAS - DS  # 36 - 28 = 8 piksel (misal offset y, x di kisaran [0, 8])

    for i in range(n):
        # 1. Tentukan jumlah pakaian untuk sampel gambar ke-i
        k_items = rng.randint(k_min, k_max + 1)
        
        # 2. Pilih indeks pakaian secara acak dari dataset sumber
        idx = rng.randint(0, len(imgs), size=k_items)
        
        for k, j in enumerate(idx):
            # 3. Acak posisi offset (y, x) di dalam kanvas
            pos_y = rng.randint(0, max_offset + 1)
            pos_x = rng.randint(0, max_offset + 1)
            
            # Tumpuk gambar pakaian (aditif maksimum 255)
            patch = X[i, pos_y:pos_y+DS, pos_x:pos_x+DS].astype(np.uint16) + imgs[j].astype(np.uint16)
            X[i, pos_y:pos_y+DS, pos_x:pos_x+DS] = np.minimum(patch, 255).astype(np.uint8)
            
            # Simpan Label & Koordinat Posisi
            Y[i, k] = labels[j]
            P[i, k] = [pos_y, pos_x]
            
    return X, Y, P

def crop_pad(X, P, k):
    out = np.zeros_like(X)
    off = (CANVAS - DS) // 2
    for i in range(len(X)):
        y, x = P[i, k]
        out[i, off:off+DS, off:off+DS] = X[i, y:y+DS, x:x+DS]
    return out

def pad_fashion(imgs):
    out = np.zeros((len(imgs), CANVAS, CANVAS), np.uint8)
    off = (CANVAS - DS) // 2
    out[:, off:off+DS, off:off+DS] = imgs
    return out

# Xg_tr, Yg_tr, Pg_tr = make_multi(fa_tr_x, fa_tr_y, CFG['N_TRAIN_MULTI'], seed=1)
# Xg_val, Yg_val, Pg_val = make_multi(fa_val_x, fa_val_y, CFG['N_VAL_MULTI'], seed=2)
# Xg_te, Yg_te, Pg_te = make_multi(fa_te_x, fa_te_y, CFG['N_TEST_MULTI'], seed=3)

# Ganti pemanggilan make_multi lama dengan make_multi_flexible
Xg_tr, Yg_tr, Pg_tr = make_multi_flexible(fa_tr_x, fa_tr_y, CFG['N_TRAIN_MULTI'], k_min=2, k_max=4, seed=1)
Xg_val, Yg_val, Pg_val = make_multi_flexible(fa_val_x, fa_val_y, CFG['N_VAL_MULTI'], k_min=2, k_max=4, seed=2)
Xg_te, Yg_te, Pg_te   = make_multi_flexible(fa_te_x, fa_te_y, CFG['N_TEST_MULTI'], k_min=2, k_max=4, seed=3)

Xs_c, Ys_c = pad_fashion(fa_tr_x), fa_tr_y
n_h = (CFG['N_SPEC_HARD'] + K - 1) // K
Xh_raw, Yh_raw, Ph_raw = make_multi(fa_tr_x, fa_tr_y, n_h, seed=4)
Xs_h = np.concatenate([crop_pad(Xh_raw, Ph_raw, k) for k in range(K)])[:CFG['N_SPEC_HARD']]
Ys_h = np.concatenate([Yh_raw[:, k] for k in range(K)])[:CFG['N_SPEC_HARD']]
nS = min(len(Xs_c), len(Xs_h))
Xs_c, Ys_c, Xs_h, Ys_h = Xs_c[:nS], Ys_c[:nS], Xs_h[:nS], Ys_h[:nS]

Xs_c_val, Ys_c_val = pad_fashion(fa_val_x), fa_val_y
Xs_h_val = np.concatenate([crop_pad(Xg_val, Pg_val, k) for k in range(K)])
Ys_h_val = np.concatenate([Yg_val[:, k] for k in range(K)])

Xs_c_te, Ys_c_te = pad_fashion(fa_te_x), fa_te_y
Xs_h_te = np.concatenate([crop_pad(Xg_te, Pg_te, k) for k in range(K)])
Ys_h_te = np.concatenate([Yg_te[:, k] for k in range(K)])

def as_dev(a):
    return torch.from_numpy(np.ascontiguousarray(a)).to(dev)

Xg_tr_t, Yg_tr_t = as_dev(Xg_tr), as_dev(Yg_tr)
Xg_val_t, Yg_val_t = as_dev(Xg_val), as_dev(Yg_val)
Xg_te_t, Yg_te_t = as_dev(Xg_te), as_dev(Yg_te)

Xs_c_t, Ys_c_t, Xs_h_t, Ys_h_t = map(as_dev, (Xs_c, Ys_c, Xs_h, Ys_h))
Xs_c_val_t, Ys_c_val_t, Xs_h_val_t, Ys_h_val_t = map(as_dev, (Xs_c_val, Ys_c_val, Xs_h_val, Ys_h_val))
Xs_c_te_t, Ys_c_te_t, Xs_h_te_t, Ys_h_te_t = map(as_dev, (Xs_c_te, Ys_c_te, Xs_h_te, Ys_h_te))

spec_g = torch.Generator().manual_seed(CFG['SEED'])
spec_lab_mask = (torch.rand(len(Xs_c_t), generator=spec_g) < CFG['LABELED_FRAC']).to(dev)

# Plot Visualisasi Data Komposit & Clean
fig, ax = plt.subplots(2, 5, figsize=(12, 5))
for i in range(5):
    ax[0, i].imshow(Xg_tr[i], cmap='gray', vmin=0, vmax=255)
    ax[0, i].set_title(f'stack: {Yg_tr[i,0]} / {Yg_tr[i,1]}', fontsize=10)
    ax[0, i].axis('off')
    ax[1, i].imshow(Xs_c[i], cmap='gray', vmin=0, vmax=255)
    ax[1, i].set_title(CLASS_NAMES[Ys_c[i]], fontsize=10)
    ax[1, i].axis('off')
plt.suptitle('Generalist inputs (top) and specialist clean inputs (bottom)')
plt.tight_layout()
plt.show()

## 3. Model Definitions

In [ ]:
class BinarySpecialistNet(nn.Module):
    def __init__(self, arch='resnet18'):
        super().__init__()
        bb = getattr(models, arch)(weights=None)
        bb.conv1 = nn.Conv2d(1, 64, kernel_size=3, stride=1, padding=1, bias=False)
        bb.maxpool = nn.Identity()
        feat = bb.fc.in_features
        bb.fc = nn.Linear(feat, 1) # Output single logit untuk binary classification
        self.backbone = bb
        
    def forward(self, x):
        return self.backbone(x).squeeze(-1) # Shape: (B,)

class MTLNet(nn.Module):
    def __init__(self, arch, n_heads, n_cls=10):
        super().__init__()
        bb = getattr(models, arch)(weights=None)
        bb.conv1 = nn.Conv2d(1, 64, kernel_size=3, stride=1, padding=1, bias=False)
        bb.maxpool = nn.Identity()
        feat = bb.fc.in_features
        bb.fc = nn.Identity()
        self.backbone = bb
        self.heads = nn.ModuleList([nn.Linear(feat, n_cls) for _ in range(n_heads)])

    def forward(self, x):
        f = self.backbone(x)
        return [head(f) for head in self.heads]

def prep(xb, shift=0):
    x = xb.float().div(255).unsqueeze(1)
    if shift:
        sy = random.randint(-shift, shift)
        sx = random.randint(-shift, shift)
        x = F.pad(x, (shift, shift, shift, shift))
        x = x[:, :, shift + sy:shift + sy + CANVAS, shift + sx:shift + sx + CANVAS]
    return x

@torch.no_grad()
def predict_binary(model, X, bs=512, temp=1.0):
    model.eval()
    probs = []
    for s in range(0, len(X), bs):
        logits = model(prep(X[s:s+bs]))
        p = torch.sigmoid(logits / temp).cpu()
        probs.append(p)
    return torch.cat(probs)

@torch.no_grad()
def predict_gen(model, X, bs=512, temp=1.0):
    model.eval()
    outs = None
    for s in range(0, len(X), bs):
        o = model(prep(X[s:s+bs]))
        o = [F.softmax(z / temp, dim=-1).cpu() for z in o]
        outs = [[a] for a in o] if outs is None else [p + [a] for p, a in zip(outs, o)]
    return [torch.cat(p) for p in outs]

## 4. Multi-Objective Gradient Methods & Pareto Utilities

In [ ]:
def flat_grads(losses, params):
    gs = []
    for l in losses:
        g = torch.autograd.grad(l, params, retain_graph=True, allow_unused=True)
        gs.append(torch.cat([(gi if gi is not None else torch.zeros_like(p)).flatten() for gi, p in zip(g, params)]))
    return torch.stack(gs)

def pcgrad(G):
    T = len(G)
    out = G.clone()
    for i in range(T):
        for j in random.sample(range(T), T):
            if j == i: continue
            dot = (out[i] * G[j]).sum()
            if dot < 0:
                out[i] = out[i] - dot / (G[j].norm() ** 2 + 1e-12) * G[j]
    return out.sum(0)

def mgda_weights(G, iters=50):
    M = G @ G.T
    T = len(G)
    a = torch.ones(T, device=G.device) / T
    for _ in range(iters):
        i = torch.argmin(M @ a)
        e = torch.zeros_like(a)
        e[i] = 1
        d = e - a
        gamma = torch.clamp(-(d @ M @ a) / (d @ M @ d + 1e-12), 0, 1)
        a = a + gamma * d
    return a

class FAMO:
    def __init__(self, T, pref=None, lr=0.025, gamma=1e-3):
        init = torch.log(torch.tensor(pref if pref is not None else [1.0 / T] * T, dtype=torch.float32))
        self.w = init.clone().to(dev).requires_grad_(True)
        self.opt = torch.optim.Adam([self.w], lr=lr, weight_decay=gamma)
        self.eps = 1e-8
        self.min_loss = torch.zeros(T, device=dev)
    def weighted(self, losses):
        D = losses - self.min_loss + self.eps
        z = F.softmax(self.w, -1)
        c = (z / D).sum().detach()
        return (D.log() * z / c).sum()
    def update(self, prev, curr):
        delta = (prev - self.min_loss + self.eps).log() - (curr - self.min_loss + self.eps).log()
        with torch.enable_grad():
            d = torch.autograd.grad(F.softmax(self.w, -1), self.w, grad_outputs=delta.detach())[0]
        self.opt.zero_grad()
        self.w.grad = d
        self.opt.step()

def train_mtl(model, loss_fn, n_train, T, method, pref, epochs, lr=CFG['LR'], bs=CFG['BS']):
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    steps = epochs * (n_train // bs)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, lr, total_steps=steps)
    shared = list(model.backbone.parameters())
    pref_t = torch.tensor(pref, device=dev, dtype=torch.float32) * T
    famo = FAMO(T, pref) if method == 'famo' else None
    
    for ep in range(epochs):
        model.train()
        perm = torch.randperm(n_train, device=dev)
        for s in range(0, n_train - bs + 1, bs):
            idx = perm[s:s+bs]
            losses = loss_fn(model, idx)
            opt.zero_grad()
            if method == 'ls':
                sum(pref_t[i] * losses[i] for i in range(T)).backward()
            elif method == 'famo':
                famo.weighted(torch.stack(losses)).backward()
            else: # pcgrad / mgda
                wl = [pref_t[i] * losses[i] for i in range(T)]
                G = flat_grads(wl, shared)
                sum(wl).backward()
                g = pcgrad(G) if method == 'pcgrad' else (mgda_weights(G) @ G) * T
                o = 0
                for p in shared:
                    n = p.numel()
                    p.grad = g[o:o+n].view_as(p).clone()
                    o += n
            opt.step()
            sched.step()
            if famo is not None:
                with torch.no_grad():
                    new = torch.stack(loss_fn(model, idx))
                famo.update(torch.stack(losses).detach(), new)
    return model

def make_prefs(T, n_extra=0, seed=0):
    prefs = [[1.0 / T] * T]
    for k in range(T):
        prefs.append([0.6 if i == k else 0.4 / (T - 1) for i in range(T)])
    rng = np.random.RandomState(seed)
    for _ in range(n_extra): prefs.append(list(rng.dirichlet(np.ones(T))))
    return prefs

def nondominated(E):
    E = np.array(E)
    mask = np.ones(len(E), bool)
    for i in range(len(E)):
        for j in range(len(E)):
            if i != j and np.all(E[j] <= E[i]) and np.any(E[j] < E[i]):
                mask[i] = False
                break
    return mask

def pareto_explore(run_fn, methods, prefs, tag=''):
    res = []
    for m in methods:
        for p in prefs:
            t0 = time.time()
            r = run_fn(m, p)
            r.update(method=m, pref=p)
            res.append(r)
            print(f'[{tag}] {m:7s} pref={np.round(p,2)} err={np.round(r["err"],4)} ({time.time()-t0:.0f}s)')
    return res

def pareto_expand(res, run_fn, rounds=1, tag=''):
    for rd in range(rounds):
        mask = nondominated([r['err'] for r in res])
        front = [r for r, k in zip(res, mask) if k]
        new = []
        for m in set(r['method'] for r in front):
            fm = sorted([r for r in front if r['method'] == m], key=lambda r: r['err'][0])
            for a, b in zip(fm[:-1], fm[1:]):
                p = ((np.array(a['pref']) + np.array(b['pref'])) / 2).tolist()
                if not any(np.allclose(p, r['pref']) and r['method'] == m for r in res):
                    new.append((m, p))
        print(f'[{tag}] expansion round {rd+1}: {len(new)} new preference points')
        for m, p in new:
            r = run_fn(m, p)
            r.update(method=m, pref=p, expanded=True)
            res.append(r)
            print(f'[{tag}] {m:7s} pref={np.round(p,2)} err={np.round(r["err"],4)} (expanded)')
    return res

def plot_pareto(res, names, title):
    E = np.array([r['err'] for r in res])
    mask = nondominated(E)
    T = E.shape[1]
    pairs = [(i, j) for i in range(T) for j in range(i + 1, T)]
    fig, axs = plt.subplots(1, len(pairs), figsize=(5.2 * len(pairs), 4.4))
    axs = np.atleast_1d(axs)
    cols = dict(ls='tab:gray', pcgrad='tab:blue', mgda='tab:green', famo='tab:red')
    for ax, (i, j) in zip(axs, pairs):
        for m in cols:
            sel = [k for k, r in enumerate(res) if r['method'] == m]
            if sel: 
                ax.scatter(E[sel, i], E[sel, j], c=cols[m], label=m, 
                           s=[60 if res[k].get('expanded') else 35 for k in sel], marker='o', alpha=.8)
        F_ = E[mask][:, [i, j]]
        F_ = F_[np.argsort(F_[:, 0])]
        ax.plot(F_[:, 0], F_[:, 1], 'k--', lw=1, label='Pareto front')
        ax.set_xlabel(f'error {names[i]}')
        ax.set_ylabel(f'error {names[j]}')
        ax.grid(alpha=.3)
    axs[0].legend()
    fig.suptitle(title)
    plt.tight_layout()
    plt.show()
    return mask

## 5. Stage 1: Pelatihan 10 Binary Specialists (One-vs-All)

In [ ]:
def train_binary_specialist(target_class, epochs=CFG['SPEC_EPOCHS'], lr=CFG['LR'], bs=CFG['BS']):
    set_seed(CFG['SEED'] + target_class)
    model = BinarySpecialistNet('resnet18').to(dev)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    
    lab_indices = torch.where(spec_lab_mask)[0]
    n_lab = len(lab_indices)
    steps = epochs * (n_lab // bs + 1)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, lr, total_steps=steps)
    # criterion = nn.BCEWithLogitsLoss()
    criterion = nn.CrossEntropyLoss(ignore_index=-1)
    
    for ep in range(epochs):
        model.train()
        perm = torch.randperm(n_lab, device=dev)
        for s in range(0, n_lab, bs):
            b_idx = lab_indices[perm[s:s+bs]]
            out_c = model(prep(Xs_c_t[b_idx], shift=2))
            out_h = model(prep(Xs_h_t[b_idx], shift=2))
            
            target_c = (Ys_c_t[b_idx] == target_class).float()
            target_h = (Ys_h_t[b_idx] == target_class).float()
            
            loss = 0.5 * criterion(out_c, target_c) + 0.5 * criterion(out_h, target_h)
            opt.zero_grad()
            loss.backward()
            opt.step()
            sched.step()
    return model

print('Training 10 One-vs-All Binary Specialists on labeled dataset split...')
t0 = time.time()
specialists = []
for cls_id in range(N_CLASSES):
    spec_m = train_binary_specialist(cls_id)
    specialists.append(spec_m)
    print(f'Specialist {cls_id} ({CLASS_NAMES[cls_id]}) trained. ({time.time()-t0:.1f}s)')

## 6. Stage 2: Generasi Soft Targets dari 10 Binary Specialists (KD)

In [ ]:
def get_specialist_soft_targets(X, P, temp=CFG['T_KD']):
    n = len(X)
    soft_targets = np.zeros((n, K, N_CLASSES), dtype=np.float32)
    for k in range(K):
        crops = as_dev(crop_pad(X, P, k))
        bin_probs = []
        for cls_id in range(N_CLASSES):
            p = predict_binary(specialists[cls_id], crops, temp=temp).numpy()
            bin_probs.append(p)
        bin_probs = np.stack(bin_probs, axis=1) # Shape: (n, 10)
        # Normalisasi probabilitas biner menjadi distribusi multi-class untuk Knowledge Distillation
        soft_dist = bin_probs / (bin_probs.sum(axis=1, keepdims=True) + 1e-8)
        soft_targets[:, k, :] = soft_dist
    return torch.from_numpy(soft_targets).to(dev)

print('Generating specialist soft targets for training composites...')
soft_t = get_specialist_soft_targets(Xg_tr, Pg_tr, temp=CFG['T_KD'])

## 7. Stage 3 & 4: Generalist Multi-Task Training, Pareto Exploration & Expansion

In [ ]:
g_gen = torch.Generator().manual_seed(123)
gen_lab_mask = (torch.rand(len(Xg_tr), generator=g_gen) < CFG['LABELED_FRAC']).to(dev)

def make_gen_loss(lam):
    def gen_loss_fn(model, idx):
        out = model(prep(Xg_tr_t[idx]))
        labeled = gen_lab_mask[idx]
        losses = []
        for k in range(K):
            kd = F.kl_div(F.log_softmax(out[k] / CFG['T_KD'], -1), soft_t[idx, k], reduction='batchmean') * (CFG['T_KD'] ** 2)
            ce = F.cross_entropy(out[k][labeled], Yg_tr_t[idx, k][labeled]) if labeled.any() else out[k].sum() * 0
            losses.append(lam * kd + (1 - lam) * ce)
        return losses
    return gen_loss_fn

def gen_eval(model, split='val'):
    X, Y = (Xg_val_t, Yg_val_t) if split == 'val' else (Xg_te_t, Yg_te_t)
    probs = predict_gen(model, X)
    return [1 - (probs[k].argmax(1).to(dev) == Y[:, k]).float().mean().item() for k in range(K)]

def gen_run(method, pref, lam=None, eval_test=False):
    set_seed(CFG['SEED'])
    # m = MTLNet('resnet50', K).to(dev)
    # Inisialisasi MTLNet sesuai jumlah head maksimal (misal K_MAX = 4)
    m = MTLNet('resnet50', n_heads=K_MAX).to(dev)
    train_mtl(m, make_gen_loss(CFG['LAM'] if lam is None else lam), len(Xg_tr_t), K, method, pref, CFG['GEN_EPOCHS'])
    result = dict(err=gen_eval(m, 'val'))
    if eval_test:
        result['test_err'] = gen_eval(m, 'test')
    return result

gen_prefs = make_prefs(K, CFG['N_PREF_EXTRA'])
gen_res = pareto_explore(gen_run, CFG['METHODS'], gen_prefs, 'generalist')
gen_res = pareto_expand(gen_res, gen_run, CFG['EXPAND_ROUNDS'], 'generalist')
gmask = plot_pareto(gen_res, ['top-left clothing', 'bottom-right clothing'], 'Generalist ResNet50: validation Pareto candidates')

## 8. Model Selection, Retrain & Final Evaluation Summary

In [ ]:
selected_candidates = [r for r, ok in zip(gen_res, gmask) if ok]
selected = min(selected_candidates, key=lambda r: np.mean(r['err']))
print('Selected Generalist (Validation):', selected['method'], np.round(selected['pref'], 2))

final_gen = gen_run(selected['method'], selected['pref'], eval_test=True)
final_base = gen_run('ls', [1.0 / K] * K, lam=0.0, eval_test=True)

final_rows = [
    dict(model='Selected Generalist (10-Binary KD)', mean_test_err=float(np.mean(final_gen['test_err'])), slot0=final_gen['test_err'][0], slot1=final_gen['test_err'][1]),
    dict(model='Baseline (Labeled-only CE)', mean_test_err=float(np.mean(final_base['test_err'])), slot0=final_base['test_err'][0], slot1=final_base['test_err'][1])
]
print('\nFINAL TEST SUMMARY:')
print(pd.DataFrame(final_rows).round(4).to_string(index=False))